In [ ]:
import time
import numpy as np
import h5py
import traceback
import pyFAI
from pyFAI.geometryRefinement import GeometryRefinement
from scipy.ndimage import median_filter, gaussian_filter
from skimage.feature import peak_local_max
from tqdm import tqdm

# --- 1. INSTRUMENT SETTINGS ---
calibrant = pyFAI.calibrant.CALIBRANT_FACTORY("alpha_Al2O3") 
calibrant.wavelength = 0.5121261413149675e-10  # 0.512 Angstroms
detector = pyFAI.detector_factory("Eiger2Cdte_1M")
# print(calibrant.wavelength)

# --- 2. OPTIMIZED PEAK PARAMETERS (Static) ---
PEAK_MIN_DIST = 2
PEAK_THRESH_REL = 0.15
PEAK_TOL_DEG = 1.0
EXCLUDE_BORDER = 20

# --- 3. THE "BLIND" GUESS ---
# Set to the mathematical center of your dataset's variance
BLIND_GUESS = [
    0.12256193161010742,  # dist (m)
    0.04503089562058449,  # poni1 (m)
    0.037385623902082444, # poni2 (m)
    0.0,                  # rot1 (rad)
    0.0,                  # rot2 (rad)
    0.0                   # rot3 (rad)
]

def run_pyfai_shootout(raw_image, ground_truth, initial_guess):
    """
    Executes a single-pass geometry refinement using robust CV pre-processing 
    and vectorized ring matching derived from the production PeakOptimizer.
    """
    start_time = time.perf_counter()
    calibrant.wavelength=0.5121261413149675e-10  # 0.512 Angstroms
    
    # --- 1. Robust Pre-Processing ---
    image_2d = np.squeeze(raw_image).astype(np.float32)
    image_2d = np.nan_to_num(image_2d, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Hard Masking (Eiger2 Gaps & Beamstop)
    image_2d[510:555, :] = 0
    image_2d[:, 510:525] = 0
    image_2d[550:, 200:280] = 0

    # Background Subtraction & Zinger Removal
    img_clean = median_filter(image_2d, size=3)
    bkg = gaussian_filter(img_clean, sigma=10)
    img_rings = img_clean - bkg
    img_rings[img_rings < 0] = 0  
    
    valid_mask = img_rings > 0
    if np.any(valid_mask):
        img_rings = np.clip(img_rings, 0, np.percentile(img_rings[valid_mask], 99.5))

    try:
        # --- 2. Peak Extraction ---
        peaks = peak_local_max(
            img_rings, 
            min_distance=PEAK_MIN_DIST, 
            threshold_rel=PEAK_THRESH_REL,
            exclude_border=EXCLUDE_BORDER
        )
        
        if len(peaks) < 15:
            return None, None, 0, False

        # --- 3. Initial Geometry Projection ---
        guess_geo = pyFAI.geometry.Geometry(
            dist=initial_guess[0], poni1=initial_guess[1], poni2=initial_guess[2], 
            rot1=initial_guess[3], rot2=initial_guess[4], rot3=initial_guess[5],
            detector=detector, wavelength=0.5121261413149675e-10
        )

        # print(calibrant.wavelength)

        # --- 4. Vectorized Ring Assignment ---
        tth_measured = guess_geo.tth(peaks[:, 0], peaks[:, 1])
        tth_expected = np.array(calibrant.get_2th())
        
        diff_matrix = np.abs(tth_measured[:, None] - tth_expected[None, :])
        min_diffs = diff_matrix.min(axis=1)
        ring_indices = diff_matrix.argmin(axis=1)
        
        mask = min_diffs < np.deg2rad(PEAK_TOL_DEG)
        
        valid_peaks = peaks[mask]
        valid_indices = ring_indices[mask]
        
        if len(np.unique(valid_indices)) < 3 or len(valid_peaks) < 10:
            return None, None, 0, False 

        data = np.column_stack((valid_peaks[:, 0], valid_peaks[:, 1], valid_indices))

        # --- 5. Progressive NLLS Refinement ---
        refiner = GeometryRefinement(
            data=data,  
            dist=guess_geo.dist, poni1=guess_geo.poni1, poni2=guess_geo.poni2,
            rot1=guess_geo.rot1, rot2=guess_geo.rot2, rot3=guess_geo.rot3,
            pixel1=detector.pixel1, pixel2=detector.pixel2,
            detector=detector, calibrant=calibrant, wavelength=0.5121261413149675e-10
        )

        # print(calibrant.wavelength)
        
        # Staged parameter release for maximum convergence stability
        refiner.refine2(fix=["poni1", "poni2", "rot1", "rot2", "rot3", "wavelength"])
        refiner.refine2(fix=["rot1", "rot2", "rot3", "wavelength"])
        refiner.refine2(fix=["rot3", "wavelength"]) 
        
        end_time = time.perf_counter()
        
        predictions = np.array([refiner.dist, refiner.poni1, refiner.poni2, refiner.rot1, refiner.rot2, refiner.rot3])
        truths = np.array([ground_truth[0], ground_truth[1], ground_truth[2], ground_truth[3], ground_truth[4], ground_truth[5]])
        
        return predictions, truths, (end_time - start_time) * 1000, True

    except Exception:
        # Fails safely without crashing the execution loop
        traceback.print_exc()
        return None, None, 0, False
    
# --- Load Dataset ---
HDF5_PATH = ""

with h5py.File(HDF5_PATH, "r") as f:
    test_dataset = [(f["test"]["images"][i], f["test"]["labels"][i]) for i in range(len(f["test"]["images"]))]

print(f"Loaded test dataset with {len(test_dataset)} images.")

# --- Initialize Metrics ---
oracle_preds, oracle_truths, oracle_times = [], [], []
blind_preds, blind_truths, blind_times = [], [], []

# --- Run Shootout ---
for raw_image, ground_truth_array in tqdm(test_dataset, desc="Evaluating NLLS Solvers"):
    
    # 1. Oracle Test (Ground Truth Initialization)
    o_preds, o_truths, o_time, o_success = run_pyfai_shootout(raw_image, ground_truth_array, ground_truth_array)
    if o_success:
        oracle_preds.append(o_preds)
        oracle_truths.append(o_truths)
        oracle_times.append(o_time)
        
    # 2. Blind Test (Dataset Mean Initialization)
    b_preds, b_truths, b_time, b_success = run_pyfai_shootout(raw_image, ground_truth_array, BLIND_GUESS)
    if b_success:
        blind_preds.append(b_preds)
        blind_truths.append(b_truths)
        blind_times.append(b_time)

# --- Print Final Benchmarks ---
total_images = len(test_dataset)

print("\n=== THE ORACLE NLLS TEST (Human-Guided) ===")
print(f"Success Rate:    {(len(oracle_preds)/total_images)*100:.1f}%")
if oracle_preds:
    print(f"Mean Latency:    {np.mean(oracle_times):.2f} ms")
    print(f"Mean Abs Error:  {np.mean([np.mean(np.abs(p - t)) for p, t in zip(oracle_preds, oracle_truths)]):.6f}")

print("\n=== THE BLIND NLLS TEST (Fully Autonomous) ===")
print(f"Success Rate:    {(len(blind_preds)/total_images)*100:.1f}%")
if blind_preds:
    print(f"Mean Latency:    {np.mean(blind_times):.2f} ms")
    print(f"Mean Abs Error:  {np.mean([np.mean(np.abs(p - t)) for p, t in zip(blind_preds, blind_truths)]):.6f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

PARAM_NAMES = ['dist', 'poni1', 'poni2', 'rot1', 'rot2', 'rot3']
PARAM_UNITS = ['m', 'm', 'm', 'rad', 'rad', 'rad']

oracle_truths = np.array(oracle_truths)
oracle_preds = np.array(oracle_preds)
blind_truths = np.array(blind_truths)
blind_preds = np.array(blind_preds)

print("pyFAI Oracle Performance Metrics")
print("-" * 50)
for i, name in enumerate(PARAM_NAMES):
    mae = mean_absolute_error(oracle_truths[:, i], oracle_preds[:, i])
    r2 = r2_score(oracle_truths[:, i], oracle_preds[:, i])
    print(f"{name:<6} (unit: {PARAM_UNITS[i]:<4}) | MAE: {mae:.6f} | R2: {r2:.4f}")

print("pyFAI Blind Performance Metrics")
print("-" * 50)
for i, name in enumerate(PARAM_NAMES):
    mae = mean_absolute_error(blind_truths[:, i], blind_preds[:, i])
    r2 = r2_score(blind_truths[:, i], blind_preds[:, i])
    print(f"{name:<6} (unit: {PARAM_UNITS[i]:<4}) | MAE: {mae:.6f} | R2: {r2:.4f}")

In [ ]:
# Predicted vs true value correlations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('pyFAI Oracle: Predicted vs. True Geometric Parameters', fontsize=16)
axes = axes.flatten()

for i, (ax, name) in enumerate(zip(axes, PARAM_NAMES)):
    ax.scatter(oracle_truths[:, i], oracle_preds[:, i], alpha=0.5, s=10, color='blue')

    # Add y=x line for reference
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()]),
    ]
    ax.plot(lims, lims, 'r--', alpha=0.75, zorder=0)
    ax.set_aspect('equal', adjustable='box')

    ax.set_title(name)
    ax.set_xlabel(f"True Value ({PARAM_UNITS[i]})")
    ax.set_ylabel(f"Predicted Value ({PARAM_UNITS[i]})")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Predicted vs true value correlations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('pyFAI Blind: Predicted vs. True Geometric Parameters', fontsize=16)
axes = axes.flatten()

for i, (ax, name) in enumerate(zip(axes, PARAM_NAMES)):
    ax.scatter(blind_truths[:, i], blind_preds[:, i], alpha=0.5, s=10, color='blue')

    # Add y=x line for reference
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()]),
    ]
    ax.plot(lims, lims, 'r--', alpha=0.75, zorder=0)
    ax.set_aspect('equal', adjustable='box')

    ax.set_title(name)
    ax.set_xlabel(f"True Value ({PARAM_UNITS[i]})")
    ax.set_ylabel(f"Predicted Value ({PARAM_UNITS[i]})")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Error distribution
errors_np = preds_oracle - labels_oracle

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('pyFAI Oracle: Error Distribution', fontsize=16)
axes = axes.flatten()

for i, (ax, name) in enumerate(zip(axes, PARAM_NAMES)):
    sns.histplot(errors_np[:, i], kde=True, ax=ax, bins=30, color='orange')
    ax.axvline(0, color='r', linestyle='--')
    ax.set_title(name)
    ax.set_xlabel(f"Prediction Error ({PARAM_UNITS[i]})")
    ax.set_ylabel("Frequency")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Error distribution
errors_np = preds_blind - labels_blind

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('pyFAI Blind: Error Distribution', fontsize=16)
axes = axes.flatten()

for i, (ax, name) in enumerate(zip(axes, PARAM_NAMES)):
    sns.histplot(errors_np[:, i], kde=True, ax=ax, bins=30, color='orange')
    ax.axvline(0, color='r', linestyle='--')
    ax.set_title(name)
    ax.set_xlabel(f"Prediction Error ({PARAM_UNITS[i]})")
    ax.set_ylabel("Frequency")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()